In [66]:
import re
import requests
import geocoder
import time
import webbrowser


In [67]:
url = 'http://127.0.0.1:1234/v1/chat/completions'

In [68]:
items = []

print('')

while True:
    item = input("welke gerechten heb je al/ heb je die je wilt gebruiken. (type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

In [69]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

In [ ]:
prompts = [
    f"""
Genereer EXACT 5 verschillende gerechten op basis van deze ingrediënten:

{items}

BELANGRIJKE REGELS:
- Geef EXACT 5 regels terug.
- Geen uitleg voor of na de regels.
- Geen markdown.
- Geen opsommingstekens.
- Elke regel moet EXACT dit formaat hebben:

gerecht_nr | gerecht | ingredienten | duur | nieuw ingredient = (ingredient)

VOORBEELD:
1 | Pasta met kip | kip, pasta, ui | 25 min | nieuw ingredient = (parmezaanse kaas)

EXTRA REGELS:
- Gebruik zoveel mogelijk ingrediënten uit de opgegeven lijst.
- Maximaal 1 nieuw ingrediënt per gerecht.
- Bereidingstijd maximaal 30 minuten.
- Het nieuwe ingrediënt MOET tussen haakjes staan.
- De tekst 'nieuw ingredient = (...)' moet altijd het laatste onderdeel van de regel zijn.
- Gebruik het pipe-teken '|' uitsluitend als scheidingsteken.
- Gebruik geen pipe-tekens in gerechtnamen of ingrediënten.
- Nummer de gerechten van 1 t/m 5.

Geef alleen de 5 regels terug.
"""
]

In [71]:
for prompt in prompts:
    start_time = time.time()

In [72]:
data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


In [73]:
try:
    response = requests.post(url, json=data)
    response.raise_for_status()

    end_time = time.time()

    result = response.json()

    answer = result["choices"][0]["message"]["content"]
    model_name = result.get("model", "onbekend")
    response_length = len(answer)

    print(answer)

    print("\nMetadata:")
    print(f"Modelnaam: {model_name}")
    print(f"Lengte response: {response_length} karakters")
    print(f"Tijd: {end_time - start_time:.2f} seconden")
    print("-" * 50)

except requests.exceptions.RequestException as e:
    print(f"Fout bij API-aanroep: {e}")


1|Kip en zalm met broccoli|kip, zalm, broccoli|20 min|nieuw ingredient = (parmezaanse kaas)  
2|Zalm en aardappels in sla met bonen|zalm, aardappels, sla, bonen|25 min|nieuw ingredient = (basilico)  
3|Kip, zalm en broccoli opgeslagen in sla|kip, zalm, broccoli|20 min|nieuw ingredient = (peterselie)  
4|Zalm, aardappels en bonen met kip|zalm, aardappels, bonen, kip|30 min|nieuw ingredient = (mosterd)  
5|Zalm, sla, broccoli en aardappels in peterselie|zalm, sla, broccoli, aardappels|20 min|nieuw ingredient = (parmezaanse kaas)

Metadata:
Modelnaam: nvidia/nemotron-3-nano-4b
Lengte response: 533 karakters
Tijd: 14.66 seconden
--------------------------------------------------


In [78]:
for answe in answer.splitlines():
    if "nieuw ingrediënt =" in answe:
        ingredient = answe.split("nieuw ingrediënt =", 1)[1].strip()

        ingredient = ingredient.strip("()")

        url = f"https://google.com/search?q=supermarkt+voor+{ingredient}+-ai"
        answe += f"|{url}"

    print(answe)


1|Kip en zalm met broccoli|kip, zalm, broccoli|20 min|nieuw ingredient = (parmezaanse kaas)  
2|Zalm en aardappels in sla met bonen|zalm, aardappels, sla, bonen|25 min|nieuw ingredient = (basilico)  
3|Kip, zalm en broccoli opgeslagen in sla|kip, zalm, broccoli|20 min|nieuw ingredient = (peterselie)  
4|Zalm, aardappels en bonen met kip|zalm, aardappels, bonen, kip|30 min|nieuw ingredient = (mosterd)  
5|Zalm, sla, broccoli en aardappels in peterselie|zalm, sla, broccoli, aardappels|20 min|nieuw ingredient = (parmezaanse kaas)


In [75]:
destination = input('wat is je lievelings supermarkt')

In [76]:
url2 = (
    f"https://www.google.com/maps/dir/?api=1"
    f"&origin={lat},{lon}"
    f"&destination={destination}"
    f"&travelmode=driving"
)

webbrowser.open(url2)

True